In [53]:
import pandas as pd
import re

In [54]:
booking_requests_df = pd.read_csv("data/booking_requests.csv")
cleaners_df = pd.read_csv("data/cleaners.csv")
customers_df = pd.read_csv("data/customers.csv")
marketing_spend_df = pd.read_csv("data/marketing_spend.csv")

# 1. Customers

In [55]:
customers_df.head()

,customer_id,signup_date,city,acquisition_channel
0,C000192,2025-01-01,Paris,Google Ads
1,C000039,2025-01-01,Paris,Google Ads
2,C000572,2025-01-01,Lyon,Organic
3,C000504,2025-01-01,Lyon,Meta Ads
4,C000307,2025-01-01,Paris,Google Ads


In [56]:
customers_df.shape

(24662, 4)

In [57]:
customers_df = customers_df.drop_duplicates()
customers_df.shape

(24662, 4)

### No duplicated values

In [58]:
customers_df.info()
# ==> no missing values 

<class 'pandas.DataFrame'>
RangeIndex: 24662 entries, 0 to 24661
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          24662 non-null  str  
 1   signup_date          24662 non-null  str  
 2   city                 24662 non-null  str  
 3   acquisition_channel  24662 non-null  str  
dtypes: str(4)
memory usage: 770.8 KB


In [59]:
n_internal = (customers_df["acquisition_channel"] == "internal").sum()
print("Internal accounts :", n_internal)

Internal accounts : 30


In [60]:
def detect_test(ch):
    if pd.isna(ch):
        return "Missing"
    elif re.search(r"tEst", str(ch), re.IGNORECASE):
        return "Test"
    else:
        return "Real"

In [61]:
#Deleting Test accounts for customers
customers_df["customer_real_account"] = customers_df["customer_id"].apply(detect_test)
customers_df

,customer_id,signup_date,city,acquisition_channel,customer_real_account
0,C000192,2025-01-01,Paris,Google Ads,Real
1,C000039,2025-01-01,Paris,Google Ads,Real
2,C000572,2025-01-01,Lyon,Organic,Real
3,C000504,2025-01-01,Lyon,Meta Ads,Real
4,C000307,2025-01-01,Paris,Google Ads,Real
...,...,...,...,...,...
24657,C_TEST25,2025-01-15,Paris,internal,Test
24658,C_TEST26,2025-01-15,Paris,internal,Test
24659,C_TEST27,2025-01-15,Paris,internal,Test
24660,C_TEST28,2025-01-15,Paris,internal,Test


In [62]:
customers_df = customers_df[customers_df["customer_real_account"]=='Real']
customers_df = customers_df.drop('customer_real_account', axis=1)
customers_df.shape
# ==> 30 customer test

(24632, 4)

### There are 30 customers test accounts

In [63]:
#Checking for internal accounts after deleting Test accounts
customers_df = customers_df[customers_df['acquisition_channel'] != "internal"]
customers_df.shape
#No internal account anymore

(24632, 4)

In [64]:
#check that the customer_id are unique and the cities names
print('The number of unique customers : ',len(customers_df['customer_id'].unique()))
print('The cities are : ', list(customers_df['city'].unique()))

The number of unique customers :  24632
The cities are :  ['Paris', 'Lyon', 'Bordeaux', 'Lille']


In [65]:
#check if the same customer_id had more than one signup_date
signup_counts = customers_df.groupby('customer_id')['signup_date'].count()
customers_df = customers_df[customers_df['customer_id'].isin(signup_counts[signup_counts == 1].index)]
customers_df.shape
# ==> each customer_id has only one sigup date

(24632, 4)

In [66]:
#check if the same customer_id had more than one city
city_counts = customers_df.groupby('customer_id')['city'].count()
customers_df = customers_df[customers_df['customer_id'].isin(city_counts[city_counts == 1].index)]
customers_df.shape
# ==> each customer_id had only 

(24632, 4)

In [67]:
#check the signup_date
dates = pd.to_datetime(customers_df['signup_date'], format='%Y-%m-%d', errors='coerce')
customers_df = customers_df[dates.notna()]
customers_df
# ==> dates are in good format

,customer_id,signup_date,city,acquisition_channel
0,C000192,2025-01-01,Paris,Google Ads
1,C000039,2025-01-01,Paris,Google Ads
2,C000572,2025-01-01,Lyon,Organic
3,C000504,2025-01-01,Lyon,Meta Ads
4,C000307,2025-01-01,Paris,Google Ads
...,...,...,...,...
24627,C024347,2026-06-30,Bordeaux,Meta Ads
24628,C022608,2026-06-30,Paris,Organic
24629,C024561,2026-06-30,Lille,Google Ads
24630,C024436,2026-06-30,Lille,Google Ads


In [68]:
customers_df['year'] = pd.to_datetime(customers_df['signup_date']).dt.year
customers_df['month'] = pd.to_datetime(customers_df['signup_date']).dt.month
customers_df['day'] = pd.to_datetime(customers_df['signup_date']).dt.day
customers_df.head()

,customer_id,signup_date,city,acquisition_channel,year,month,day
0,C000192,2025-01-01,Paris,Google Ads,2025,1,1
1,C000039,2025-01-01,Paris,Google Ads,2025,1,1
2,C000572,2025-01-01,Lyon,Organic,2025,1,1
3,C000504,2025-01-01,Lyon,Meta Ads,2025,1,1
4,C000307,2025-01-01,Paris,Google Ads,2025,1,1


In [69]:
customers_df['acquisition_channel'].unique()

<StringArray>
['Google Ads', 'Organic', 'Meta Ads', 'Referral']
Length: 4, dtype: str

# 2. cleaners

In [70]:
cleaners_df.head()

,cleaner_id,application_date,city,acquisition_channel,onboarded_date,deactivated_date
0,P00027,2025-01-01,Paris,Meta Ads,2025-01-19,2025-01-30
1,P00002,2025-01-01,Paris,Meta Ads,2025-01-10,2025-12-31
2,P00028,2025-01-01,Paris,Job boards,2025-01-10,2025-08-24
3,P00014,2025-01-02,Paris,Meta Ads,2025-01-19,NaN
4,P00025,2025-01-04,Paris,Job boards,2025-01-23,2025-10-23


In [71]:
cleaners_df.shape

(1511, 6)

In [72]:
cleaners_df = cleaners_df.drop_duplicates()
cleaners_df.shape

(1511, 6)

### No duplicated rows

In [73]:
cleaners_df.info()
# ==> no missing values

<class 'pandas.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   cleaner_id           1511 non-null   str  
 1   application_date     1511 non-null   str  
 2   city                 1511 non-null   str  
 3   acquisition_channel  1511 non-null   str  
 4   onboarded_date       1101 non-null   str  
 5   deactivated_date     537 non-null    str  
dtypes: str(6)
memory usage: 71.0 KB


In [74]:
#Check if there are test accounts in cleaner_id
cleaners_df["cleaner_real_account"] = cleaners_df["cleaner_id"].apply(detect_test)
cleaners_df = cleaners_df[cleaners_df["cleaner_real_account"]=='Real']
cleaners_df = cleaners_df.drop('cleaner_real_account', axis=1)
cleaners_df.shape
# ==> no test accounts for cleaner

(1511, 6)

In [75]:
#check for the city names
cleaners_df["city"].unique()

<StringArray>
['Paris', 'Lyon', 'Bordeaux', 'Lille']
Length: 4, dtype: str

In [76]:
cleaners_df

,cleaner_id,application_date,city,acquisition_channel,onboarded_date,deactivated_date
0,P00027,2025-01-01,Paris,Meta Ads,2025-01-19,2025-01-30
1,P00002,2025-01-01,Paris,Meta Ads,2025-01-10,2025-12-31
2,P00028,2025-01-01,Paris,Job boards,2025-01-10,2025-08-24
3,P00014,2025-01-02,Paris,Meta Ads,2025-01-19,NaN
4,P00025,2025-01-04,Paris,Job boards,2025-01-23,2025-10-23
...,...,...,...,...,...,...
1506,P01504,2026-06-29,Lille,Meta Ads,2026-07-12,NaN
1507,P01461,2026-06-29,Bordeaux,Job boards,2026-07-13,NaN
1508,P01506,2026-06-30,Lille,Referral,NaN,NaN
1509,P01453,2026-06-30,Lyon,Job boards,2026-07-20,NaN


# 3. Booking_requests

In [77]:
booking_requests_df.head()

,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0


In [78]:
booking_requests_df['rating'].unique()

array([ 5.,  4., nan,  3.,  1.,  2.])

In [79]:
booking_requests_df["status"].unique()

<StringArray>
['fulfilled', 'unfulfilled', 'cancelled_by_customer']
Length: 3, dtype: str

In [80]:
n_init =booking_requests_df.shape

In [81]:
booking_requests_df = booking_requests_df.drop_duplicates()
print("the number of duplicated booking requests:", n_init[0]-booking_requests_df.shape[0])

the number of duplicated booking requests: 1323


In [82]:
booking_requests_df.info()

<class 'pandas.DataFrame'>
Index: 132384 entries, 0 to 133706
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   request_id    132384 non-null  str    
 1   customer_id   132384 non-null  str    
 2   city          132384 non-null  str    
 3   request_date  132384 non-null  str    
 4   hours         132384 non-null  int64  
 5   price_eur     131286 non-null  float64
 6   status        132384 non-null  str    
 7   cleaner_id    108912 non-null  str    
 8   rating        108912 non-null  float64
dtypes: float64(2), int64(1), str(6)
memory usage: 10.1 MB


In [83]:
booking_requests_df = booking_requests_df[booking_requests_df["customer_id"].apply(detect_test) == "Real"]
booking_requests_df = booking_requests_df[booking_requests_df["cleaner_id"].apply(detect_test) != "Test"]
booking_requests_df = booking_requests_df[booking_requests_df["request_id"].apply(detect_test) == "Real"]
booking_requests_df.shape
# ==> There are 220 Test IDs

(132164, 9)

### It is normal that unfulfilled or cancelled_by_customer requests don't have values in price_eur, but we can fill the price_eur for the ones that are fulfilled, by using this formula mentioned in the project description: hour * €27

In [84]:
booking_requests_df[(booking_requests_df["price_eur"].isna()) & (booking_requests_df["status"]=="fulfilled")]

,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating
176,R0060855,C006577,Paris,2026-01-06,2,NaN,fulfilled,P00440,5.0
397,R0012574,C003283,Paris,2025-06-11,3,NaN,fulfilled,P00018,5.0
547,R0007176,C000006,Paris,2025-05-26,2,NaN,fulfilled,P00007,5.0
689,R0086004,C017292,Paris,2026-03-19,3,NaN,fulfilled,P00690,5.0
732,R0056356,C010841,Lyon,2025-12-11,3,NaN,fulfilled,P00541,4.0
...,...,...,...,...,...,...,...,...,...
132816,R0055053,C003723,Lyon,2025-12-04,2,NaN,fulfilled,P00115,1.0
132934,R0102930,C016356,Bordeaux,2026-04-10,3,NaN,fulfilled,P00652,5.0
133387,R0103065,C016516,Bordeaux,2026-04-04,2,NaN,fulfilled,P01146,5.0
133637,R0058265,C011418,Bordeaux,2025-12-18,3,NaN,fulfilled,P00740,3.0


In [85]:
booking_requests_df.loc[
    (booking_requests_df['status'] == 'fulfilled') &
    (booking_requests_df['price_eur'].isnull()),
    'price_eur'
] = booking_requests_df['hours'] * 27

In [86]:
booking_requests_df["status"].unique()

<StringArray>
['fulfilled', 'unfulfilled', 'cancelled_by_customer']
Length: 3, dtype: str

In [87]:
booking_requests_df['rating'].unique()

array([ 5.,  4., nan,  3.,  1.,  2.])

In [88]:
booking_requests_df["city"].unique()

<StringArray>
[    'Lyon', 'Bordeaux',    'Paris',    'Lille',    'PARIS', 'Paris 11',
    'lyon ',  'Lyon 3e',     'LYON']
Length: 9, dtype: str

In [89]:
#Standardize the cities
def norm_city(s):
    s = s.lower()
    if "paris" in s:
        return "Paris"
    elif "lyon" in s:
        return "Lyon"
    else:
        return s.title()

In [90]:
booking_requests_df["city"] = booking_requests_df["city"].apply(norm_city)
booking_requests_df["city"].unique()

<StringArray>
['Lyon', 'Bordeaux', 'Paris', 'Lille']
Length: 4, dtype: str

In [91]:
#Create Bool columns for fulfilled and canc
booking_requests_df["fulfilled"] = booking_requests_df["status"] == "fulfilled"
booking_requests_df["canc"] = booking_requests_df['status'] == "cancelled_by_customer"

In [92]:
#Check request_date format
def parse_date(s):
    if "-" in s:
        return pd.to_datetime(s, format="%Y-%m-%d")
    else:
        return pd.to_datetime(s, format="%d/%m/%Y")

In [93]:
booking_requests_df["request_date"] = booking_requests_df['request_date'].apply(parse_date)
#Create month starting from 01 for each month
booking_requests_df["month"] = booking_requests_df['request_date'].dt.to_period("M").dt.to_timestamp()
booking_requests_df

,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0,True,False,2025-01-01
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0,True,False,2025-05-01
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN,False,False,2025-02-01
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0,True,False,2026-01-01
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0,True,False,2025-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...
133702,R0038441,C004736,Lyon,2025-10-24,4,108.0,fulfilled,P00234,5.0,True,False,2025-10-01
133703,R0040942,C009999,Bordeaux,2025-10-25,3,81.0,fulfilled,P00170,5.0,True,False,2025-10-01
133704,R0027337,C002467,Paris,2025-09-21,2,54.0,fulfilled,P00501,5.0,True,False,2025-09-01
133705,R0106418,C010209,Paris,2026-05-20,3,81.0,fulfilled,P01096,5.0,True,False,2026-05-01


# 4. Marketing_spend

In [94]:
marketing_spend_df.head()

,month,city,side,channel,spend_eur
0,2025-01-01,Paris,demand,Google Ads,3920.0
1,2025-01-01,Paris,demand,Meta Ads,2880.0
2,2025-01-01,Paris,supply,Job boards,1620.0
3,2025-01-01,Paris,supply,Meta Ads,480.0
4,2025-01-01,Lyon,demand,Google Ads,2290.0


In [95]:
marketing_spend_df.shape

(243, 5)

In [96]:
marketing_spend_df = marketing_spend_df.drop_duplicates()
marketing_spend_df.shape
# ==> no duplicated rows

(243, 5)

In [97]:
marketing_spend_df.info()
# ==> No missing values 

<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   month      243 non-null    str    
 1   city       243 non-null    str    
 2   side       243 non-null    str    
 3   channel    243 non-null    str    
 4   spend_eur  243 non-null    float64
dtypes: float64(1), str(4)
memory usage: 9.6 KB


In [98]:
marketing_spend_df["month"] = pd.to_datetime(marketing_spend_df["month"], format="%Y-%m-%d", errors="coerce")
print("dates invalides :", marketing_spend_df["month"].isna().sum())
# ==> There is no invalid format


dates invalides : 0


In [99]:
marketing_spend_df['city'].unique()

<StringArray>
['Paris', 'Lyon', 'Bordeaux', 'Lille']
Length: 4, dtype: str

In [100]:
#Check if there are any negative values in "spend_eur"
marketing_spend_df = marketing_spend_df[marketing_spend_df["spend_eur"]>0]
marketing_spend_df.shape
# ==> There is no negative or null values ; all of the values >0

(243, 5)

In [101]:
marketing_spend_df['side'].unique()

<StringArray>
['demand', 'supply']
Length: 2, dtype: str

In [ ]:
fb_chk = booking_requests_df[booking_requests_df.fulfilled].merge(cleaners_df, on="cleaner_id", how="left", suffixes=("", "_cl"))
pre  = (fb_chk.request_date < fb_chk.onboarded_date).sum()
post = (fb_chk.request_date > fb_chk.deactivated_date).sum()
print(f"jobs before onboarding : {pre} ({pre/len(fb_chk):.1%}) | jobs after desactivation : {post}")
print("cleaners desactivated before onboarded :", int((cleaners_df.deactivated_date < cleaners_df.onboarded_date).sum()))

jobs before onboarding : 9324 (8.6%) | jobs after desactivation : 1336
cleaners désactivés avant d'être onboardés : 21
